# Brecha de precios entre cadenas (AMBA): LightGBM + SHAP

Notebook del modelo usado en la sección 2.2 del artículo «Busque, compare y después compre: los precios en el AMBA».
Se comparte para mostrar los aspectos técnicos de la predicción.

**Objetivo.** Predecir la brecha de precios de octubre de 2026 y ver qué variables pesan más en esa predicción,
separándolas en bloques: territoriales, de negocio, de cadena, dinámica y calendario.

**Lectura de SHAP.** El SHAP muestra qué variables usa el modelo para *predecir* la brecha. No es una estimación causal:
que una variable tenga mucho peso no significa que *cause* la brecha, y que tenga poco no prueba que no la afecte
(puede estar absorbida por otra variable correlacionada).

**Unidad de análisis:** sucursal × categoría × origen (nacional/importado) × mes.

**Variable a predecir (brecha de la celda, puntos log ≈ %):**
1. Para cada producto compartido (EAN vendido por más de una empresa) en la sucursal *s*:
   `g = 100 · [ln p(s) − promedio de ln p en las sucursales de las OTRAS empresas del mismo formato]`.
   Formatos: super = DIA, Carrefour Express, Carrefour Market, Disco, SuperChangomas;
   hiper = Coto, Carrefour Hiper, Jumbo, Vea, Changomas, HiperChangomas. Maxi excluido.
   Se exigen al menos 3 sucursales rivales con ese producto en el mes.
2. La brecha de la celda es el promedio de `g` sobre los productos de la celda que también estaban en t−1
   (misma canasta para la brecha de t y su rezago).

**Lanús:** sus sucursales no son observaciones ni entran en la referencia de rivales (efecto de la tasa municipal, ver
el estudio de diferencias en diferencias), pero sí se cuentan como rivales cercanos en la variable de competencia.

**Calidad de datos:** se descarta el mes de una sucursal que informa menos de la mitad de los productos de la
sucursal mediana de su bandera en ese mes; SuperChangomas San Martín (11|1037) se excluye de ago a oct-2026
(lista congelada, igual que en el IPC); precios a más de 5 veces de la mediana del producto se descartan.

**Entrenamiento:** feb-2025 a sep-2026 (enero de 2025 sólo aporta rezagos). Octubre de 2026 es el mes de prueba.

**Muestra y modelo:** 70% de las sucursales de cada bandera. Hiperparámetros fijos; 10 semillas derivadas de 91218.
SHAP sobre todas las celdas de octubre de 2026.

In [ ]:
# Configuración
PLATAFORMA = 'kaggle'    # 'kaggle', 'colab' o 'local'
FRACCION_SUCURSALES = 0.7    # 70% de las sucursales de cada bandera (sin Lanús y sin las inactivas en octubre)
N_MODELOS_SHAP = 10
SEMILLA = 91218

import os, time, json
if PLATAFORMA == 'colab':
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = '/content/drive/MyDrive/SEPA/brecha_precios/colab'
    INS, SAL = f'{BASE}/insumos', f'{BASE}/salidas'
elif PLATAFORMA == 'kaggle':
    import glob as _g
    # el dataset se sube con la carpeta 'insumos' (sucursales.parquet, productos.parquet, precios/*.parquet)
    _s = _g.glob('/kaggle/input/**/sucursales.parquet', recursive=True)
    assert _s, 'No encuentro sucursales.parquet: agregá el dataset con los insumos a la notebook'
    INS, SAL = os.path.dirname(_s[0]), '/kaggle/working/salidas'
else:
    BASE = os.environ.get('BASE_BRECHA', '.')
    INS, SAL = f'{BASE}/insumos', f'{BASE}/salidas'
print('Insumos:', INS, '| Salidas:', SAL)
os.makedirs(SAL, exist_ok=True)

import numpy as np, pandas as pd, polars as pl, lightgbm as lgb
print('lightgbm', lgb.__version__, '| polars', pl.__version__)

## 1. Calidad de datos y construcción de variables

In [ ]:
t0 = time.time()
suc = pl.read_parquet(f'{INS}/sucursales.parquet')
prod = pl.read_parquet(f'{INS}/productos.parquet')
PERIODOS = [f'{y}-{m:02d}' for y in (2025, 2026) for m in range(1, 13)][:22]   # t = 1 … 22 (ene-2025 … oct-2026)
T_PRUEBA = 22

suc = suc.with_columns(lanus=pl.col('partido') == 'Lanus')
print('Sucursales:', suc.height, '| en Lanús:', suc['lanus'].sum())
print(suc.group_by('grupo', 'empresa', 'bandera').len().sort('grupo', 'bandera'))

precios = pl.scan_parquet(f'{INS}/precios/*.parquet')   # un archivo por mes y mitad de sucursales
# control de calidad: cobertura de cada sucursal por mes. Un mes con menos de la mitad de los productos de la
# sucursal mediana de su bandera se descarta como hueco de datos (cierres, fallas de carga).
cob = precios.group_by('ck_id', 't').agg(n=pl.len()).collect().join(suc.select('ck_id', 'ck', 'bandera'), on='ck_id')
cob = cob.with_columns(med=pl.col('n').median().over('bandera', 't')).with_columns(ok=pl.col('n') >= 0.5 * pl.col('med'))
# SuperChangomas San Martín: lista congelada de ago a oct-2026 (misma exclusión que en el IPC)
cob = cob.with_columns(ok=pl.col('ok') & ~((pl.col('ck') == '11|1037') & pl.col('t').is_in([20, 21, 22])))
MESES_OK = cob.filter(pl.col('ok')).select('ck_id', 't')
print('Meses-sucursal descartados (cobertura < 50% de la mediana de su bandera, o San Martín):', cob.filter(~pl.col('ok')).height, 'de', cob.height)
activas_oct = MESES_OK.filter(pl.col('t') == T_PRUEBA)['ck_id']

# muestra de sucursales: FRACCION dentro de cada bandera, sin Lanús, activas en octubre, al menos 1 por bandera
rng = np.random.default_rng(SEMILLA)
elegibles = suc.filter(~pl.col('lanus') & pl.col('ck_id').is_in(activas_oct)).to_pandas()
muestra = []
for b, g in elegibles.groupby('bandera'):
    k = max(1, int(round(len(g) * FRACCION_SUCURSALES)))
    muestra += list(rng.choice(g.ck_id.values, size=k, replace=False))
MUESTRA = pl.Series('ck_id', np.sort(muestra), dtype=pl.UInt16)
print(f'Sucursales en la muestra: {len(MUESTRA)}')

ref = suc.filter(~pl.col('lanus')).select('ck_id', 'empresa', 'bandera', 'grupo')   # sucursales que forman la referencia
suc_obs = suc.select('ck_id', 'empresa', 'grupo')

In [ ]:
# Por mes: referencia de rivales, dispersión y brecha de cada producto en las sucursales de la muestra
G, DISP = [], []
for t in range(1, T_PRUEBA + 1):
    d = (precios.filter((pl.col('t') == t) & (pl.col('p') > 0)).with_columns(lp=pl.col('p').log())
         .select('ck_id', 'ean', 'lp').collect()
         .join(MESES_OK.filter(pl.col('t') == t).select('ck_id'), on='ck_id'))
    # control de calidad: descartar precios a más de 5 veces (o menos de 1/5) de la mediana del producto en el mes
    d = d.with_columns(med=pl.col('lp').median().over('ean')).filter((pl.col('lp') - pl.col('med')).abs() <= np.log(5)).drop('med')
    r = d.join(ref, on='ck_id')
    emp = r.group_by('grupo', 'ean', 'empresa').agg(Sc=pl.col('lp').sum(), Nc=pl.len())
    tot = emp.group_by('grupo', 'ean').agg(S=pl.col('Sc').sum(), N=pl.col('Nc').sum())
    # dispersión (rango intercuartil del ln precio, en puntos): entre cadenas (formato) e intra-cadena (bandera)
    de = r.group_by('grupo', 'ean').agg(disp_entre=100 * (pl.col('lp').quantile(.75) - pl.col('lp').quantile(.25)))
    di = r.group_by('bandera', 'ean').agg(disp_intra=100 * (pl.col('lp').quantile(.75) - pl.col('lp').quantile(.25)))
    DISP.append((t, de, di))
    o = (d.filter(pl.col('ck_id').is_in(MUESTRA)).join(suc_obs, on='ck_id')
         .join(tot, on=['grupo', 'ean']).join(emp, on=['grupo', 'ean', 'empresa'], how='left')
         .with_columns(pl.col('Sc').fill_null(0), pl.col('Nc').fill_null(0))
         .with_columns(n_rival=pl.col('N') - pl.col('Nc'))
         .filter(pl.col('n_rival') >= 3)
         .with_columns(g=(100 * (pl.col('lp') - (pl.col('S') - pl.col('Sc')) / pl.col('n_rival'))).cast(pl.Float32),
                       t=pl.lit(t, pl.Int16))
         .select('ck_id', 'ean', 't', 'g'))
    G.append(o)
    print(f't={t:2d} {PERIODOS[t-1]}  precios={d.height:>9,}  brechas producto (muestra)={o.height:>8,}', flush=True)
G = pl.concat(G)
# brechas extremas (> 100 puntos, factor ~2,7) se tratan como errores de carga
print('Brechas de producto descartadas por |g| > 100:', (G['g'].abs() > 100).sum())
G = G.filter(pl.col('g').abs() <= 100)
print(f'Paso 1: {time.time() - t0:.0f} s')

In [ ]:
# Celda = sucursal × categoría × origen. Rezagos con la misma canasta.
prod_c = prod.select('ean', pl.col('categoria').fill_null('S/C'), pl.col('importado').fill_null(0).cast(pl.Int8), 'marca')
G = G.join(prod_c, on='ean', how='left').with_columns(pl.col('categoria').fill_null('S/C'), pl.col('importado').fill_null(0))
lag = lambda k, nombre: G.select('ck_id', 'ean', (pl.col('t') + k).cast(pl.Int16).alias('t'), pl.col('g').alias(nombre))
P = G.join(lag(1, 'g1'), on=['ck_id', 'ean', 't'], how='left') \
     .join(lag(2, 'g2'), on=['ck_id', 'ean', 't'], how='left')
# dispersión del producto en t−1
de = pl.concat([x[1].with_columns(t=pl.lit(x[0] + 1, pl.Int16)) for x in DISP])
di = pl.concat([x[2].with_columns(t=pl.lit(x[0] + 1, pl.Int16)) for x in DISP])
P = (P.join(suc.select('ck_id', 'grupo', 'bandera'), on='ck_id')
      .join(de, on=['grupo', 'ean', 't'], how='left').join(di, on=['bandera', 'ean', 't'], how='left'))
P = P.filter(pl.col('g1').is_not_null())            # canasta de t = productos presentes en t y en t−1

celda = ['ck_id', 'categoria', 'importado', 't']
c2 = pl.col('g2').is_not_null()
C = P.group_by(celda).agg(
    brecha=pl.col('g').mean(),
    lag1=pl.col('g1').mean(),
    lag2=pl.col('g2').filter(c2).mean(),
    delta=(pl.col('g1') - pl.col('g2')).filter(c2).mean(),
    n_productos=pl.len(),
    conc_marca=(pl.col('marca').drop_nulls().value_counts(sort=True).struct.field('count').first() / pl.len()),
    disp_entre=pl.col('disp_entre').mean(),
    disp_intra=pl.col('disp_intra').mean(),
)
C = (C.join(suc.select('ck_id', 'bandera', 'empresa', 'grupo', 'lat', 'lon', 'decil_ingreso', 'rivales_1km'), on='ck_id')
      .with_columns(mes_del_anio=((pl.col('t') - 1) % 12 + 1).cast(pl.Int8),
                    periodo=pl.col('t').map_elements(lambda x: PERIODOS[x - 1], return_dtype=pl.String))
      .sort(celda))
C.write_parquet(f'{SAL}/celdas.parquet')
print('Celdas:', C.height, '| meses:', C['t'].min(), '→', C['t'].max())
print(C.group_by('bandera').agg(celdas=pl.len(), brecha_media=pl.col('brecha').mean().round(2),
                                productos_por_celda=pl.col('n_productos').median()).sort('bandera'))
print(f'Paso 1 completo: {time.time() - t0:.0f} s')

## 2. Variables del modelo y bloques para leer el SHAP

In [ ]:
BLOQUES = {
    'territorial': ['lat', 'lon', 'decil_ingreso', 'rivales_1km', 'ck_id'],
    'negocio':     ['categoria', 'importado', 'n_productos', 'conc_marca', 'disp_intra', 'disp_entre'],
    'cadena':      ['bandera', 'empresa', 'grupo'],
    'dinamica':    ['lag1', 'lag2', 'delta'],
    'calendario':  ['mes_del_anio'],
}
VARS = [v for b in BLOQUES.values() for v in b]
CATS = ['ck_id', 'categoria', 'bandera', 'empresa', 'grupo']
df = C.to_pandas()
for c in CATS:
    df[c] = pd.Categorical(df[c].astype(str))
dev = df[(df.t >= 2) & (df.t <= T_PRUEBA - 1)]
oct_ = df[df.t == T_PRUEBA].copy()
print('Entrenamiento:', PERIODOS[dev.t.min() - 1], '→', PERIODOS[dev.t.max() - 1], f'({dev.t.nunique()} meses)')
print('Filas: entrenamiento', len(dev), '| octubre', len(oct_))

## 3. Hiperparámetros

Se eligieron con una optimización bayesiana (4 puntos iniciales Sobol y 6 guiados, semilla 91218) validada con
walk-forward de ventana expansiva sobre los últimos meses de entrenamiento. Para evitar la maldición del ganador se usa
la **segunda mejor** combinación (RMSE walk-forward 3,086; la mejor fue 3,051).

In [ ]:
HIPER = {'min_data_in_leaf': 1141, 'feature_fraction': 0.428158870658, 'feature_fraction_bynode': 0.458764433704,
         'learning_rate': 0.008895703812, 'num_iterations': 2560, 'num_leaves': 9, 'min_sum_hessian_in_leaf': 0.954710452810}
BASE_PAR = dict(objective='regression', metric='rmse', verbose=-1, deterministic=True, force_col_wise=True,
                max_cat_to_onehot=4, cat_smooth=10)
json.dump(HIPER, open(f'{SAL}/hiperparametros.json', 'w'), indent=1)

def entrenar(par, X, y, semilla):
    p = {**BASE_PAR, **par, 'seed': semilla}
    n = p.pop('num_iterations')
    cats = [c for c in CATS if c in X.columns]
    return lgb.train(p, lgb.Dataset(X, y, categorical_feature=cats, free_raw_data=False), num_boost_round=n)

rmse = lambda y, p: float(np.sqrt(np.mean((y - p) ** 2)))
mae = lambda y, p: float(np.mean(np.abs(y - p)))

## 4. Modelo final: 10 semillas, métricas de octubre y referencias ingenuas

Se compara contra dos referencias: la persistencia (brecha de octubre = brecha de septiembre de la celda) y la media
histórica de la bandera.

In [ ]:
SEMILLAS = np.random.default_rng(SEMILLA).integers(1, 2**31 - 1, size=10)
DIR_MOD = f'{SAL}/modelos'; os.makedirs(DIR_MOD, exist_ok=True)
modelos, preds, filas = [], [], []
for i, s in enumerate(SEMILLAS):
    f = f'{DIR_MOD}/lgbm_semilla_{i}.txt'
    m = lgb.Booster(model_file=f) if os.path.exists(f) else entrenar(HIPER, dev[VARS], dev.brecha, int(s))
    if not os.path.exists(f): m.save_model(f)
    p = m.predict(oct_[VARS]); modelos.append(m); preds.append(p)
    filas.append(dict(semilla=int(s), rmse=rmse(oct_.brecha, p), mae=mae(oct_.brecha, p)))
por_semilla = pd.DataFrame(filas)
oct_['pred'] = np.mean(preds, axis=0)
oct_['pred_persistencia'] = oct_.lag1
oct_['pred_media_bandera'] = oct_.bandera.map(dev.groupby('bandera', observed=True).brecha.mean()).astype(float)

def metricas(g):
    return pd.Series({'celdas': len(g),
                      'RMSE_modelo': rmse(g.brecha, g.pred), 'MAE_modelo': mae(g.brecha, g.pred),
                      'RMSE_persistencia': rmse(g.brecha, g.pred_persistencia), 'MAE_persistencia': mae(g.brecha, g.pred_persistencia),
                      'RMSE_media_bandera': rmse(g.brecha, g.pred_media_bandera), 'MAE_media_bandera': mae(g.brecha, g.pred_media_bandera)})
tabla = pd.concat([metricas(oct_).to_frame('Total').T,
                   oct_.groupby('bandera', observed=True)[['brecha', 'pred', 'pred_persistencia', 'pred_media_bandera']].apply(metricas)]).round(3)
por_semilla.to_csv(f'{SAL}/octubre_por_semilla.csv', index=False)
tabla.to_csv(f'{SAL}/octubre_metricas.csv')
print('Octubre, por semilla: RMSE {:.3f} ± {:.3f} | MAE {:.3f} ± {:.3f}'.format(
    por_semilla.rmse.mean(), por_semilla.rmse.std(), por_semilla.mae.mean(), por_semilla.mae.std()))
print(tabla.to_string())

## 5. SHAP: composición por bloques y dirección de cada variable

Contribuciones exactas de LightGBM (TreeSHAP) sobre todas las celdas de octubre, promediadas entre las 10 semillas.
Cada parte se guarda al terminar: si la sesión se corta, al volver a correr la celda retoma.

- **Composición** (gráfico «¿Qué explica la brecha de precios de cada empresa?»): participación de cada bloque en el
  |SHAP| medio. Se excluye el bloque de dinámica (rezagos de la brecha) para concentrarse en lo estructural.
- **Dirección** (gráfico «¿Qué variables suben y cuáles bajan la brecha de cada empresa?»): SHAP medio de cada
  variable, con signo, en puntos de brecha.

Recordatorio: describe qué usa el modelo para predecir, no relaciones causales.

In [ ]:
CK = f'{SAL}/shap_checkpoints'; os.makedirs(CK, exist_ok=True)
TAM = 5000
partes = [(a, min(a + TAM, len(oct_))) for a in range(0, len(oct_), TAM)]
X_o = oct_[VARS]
for mi, m in enumerate(modelos[:N_MODELOS_SHAP]):
    for (a, b) in partes:
        f = f'{CK}/m{mi:02d}_{a:06d}.npy'
        if os.path.exists(f): continue
        t = time.time(); np.save(f, m.predict(X_o.iloc[a:b], pred_contrib=True).astype(np.float32))
        print(f'SHAP modelo {mi + 1}/{N_MODELOS_SHAP} | celdas {a}-{b} de {len(oct_)} | {time.time() - t:.0f} s', flush=True)
SH = np.mean([np.concatenate([np.load(f'{CK}/m{mi:02d}_{a:06d}.npy') for (a, b) in partes])
              for mi in range(min(N_MODELOS_SHAP, len(modelos)))], axis=0)
shap = pd.DataFrame(SH[:, :-1], columns=VARS, index=oct_.index)
shap.assign(bandera=oct_.bandera.astype(str), empresa=oct_.empresa.astype(str)).to_parquet(f'{SAL}/shap_octubre.parquet')

BLOQ_ESTRUCT = {b: v for b, v in BLOQUES.items() if b != 'dinamica'}
bloq = pd.DataFrame({b: shap[v].sum(axis=1) for b, v in BLOQ_ESTRUCT.items()})
for nivel in ('empresa', 'bandera'):
    clave = oct_[nivel].astype(str)
    a = bloq.abs().groupby(clave).mean()
    comp = (100 * a.div(a.sum(axis=1), axis=0)).round(1)
    dire = shap[[v for b in BLOQ_ESTRUCT.values() for v in b]].groupby(clave).mean().round(3)
    comp.to_csv(f'{SAL}/shap_composicion_{nivel}.csv'); dire.to_csv(f'{SAL}/shap_direccion_{nivel}.csv')
    print(f'\nComposición por {nivel} (% del |SHAP| medio, sin dinámica):'); print(comp.to_string())
    print(f'\nDirección por {nivel} (SHAP medio, puntos de brecha):'); print(dire.T.to_string())
print(f'Listo. Salidas en {SAL} | tiempo total {time.time() - t0:.0f} s')